# 批量计算台风方位角平均风场
该 Notebook 脚本用于批量处理 CM1 输出文件。它遍历所有时间步，利用平滑最小气压算法动态确定每个时刻的台风中心，计算 2000km 半径内的方位角平均风场（径向风、切向风、垂直风），并将结果保存为 NetCDF 文件。

In [1]:
# 1. 导入必要库与参数配置
import numpy as np
from netCDF4 import Dataset
import os
import sys

# 尝试导入之前单元格中提到的台风中心定位函数
# 确保 cm1_psfc_min_smoother.py 在当前工作目录或 PYTHONPATH 中
try:
    from cm1_psfc_min_smoother import find_smoothed_min_point
except ImportError:
    print("警告: 未找到 cm1_psfc_min_smoother 模块。请确保该文件存在。")
    # 定义一个占位函数以防报错（实际运行时请确保模块存在）
    def find_smoothed_min_point(nc_file, time_key, **kwargs):
        return {'x': 0.0, 'y': 0.0}

# -----------------------------------------------------------
# 参数设置
# -----------------------------------------------------------
nc_file = "dataset/cm1out.nc"
output_file = "dataset/typhoon_azimuthal_avg_dynamic.nc"

# 径向分箱设置 (单位: km)
max_r = 2000.0   # 最大半径 (根据要求修改为 2000km)
dr = 2.0         # 半径步长
r_bins = np.arange(0, max_r + dr, dr)
r_centers = 0.5 * (r_bins[:-1] + r_bins[1:]) # 输出的 r 坐标
nr = len(r_centers)

print(f"输入文件: {nc_file}")
print(f"输出文件: {output_file}")
print(f"径向范围: 0 - {max_r} km, 步长: {dr} km")

输入文件: dataset/cm1out.nc
输出文件: dataset/typhoon_azimuthal_avg_dynamic.nc
径向范围: 0 - 2000.0 km, 步长: 2.0 km


In [2]:
# 2. 初始化输出 NetCDF 文件结构
# 先读取一次输入文件获取固定维度信息
with Dataset(nc_file, 'r') as nc_in:
    zh = nc_in.variables['zh'][:]
    nz = len(zh)
    # 获取总时间步数
    total_times = len(nc_in.variables['time'])
    print(f"检测到总时间步数: {total_times}, 高度层数: {nz}")

# 如果输出文件已存在，先删除
if os.path.exists(output_file):
    os.remove(output_file)

# 创建输出文件并定义结构
nc_out = Dataset(output_file, 'w', format='NETCDF4')

# 定义维度
# time 定义为 unlimited，方便逐行追加
nc_out.createDimension('time', None) 
nc_out.createDimension('z', nz)
nc_out.createDimension('r', nr)

# 创建变量
var_time = nc_out.createVariable('time', 'f4', ('time',))
var_z = nc_out.createVariable('z', 'f4', ('z',))
var_r = nc_out.createVariable('r', 'f4', ('r',))

var_u = nc_out.createVariable('u', 'f4', ('time', 'z', 'r'))
var_v = nc_out.createVariable('v', 'f4', ('time', 'z', 'r'))
var_w = nc_out.createVariable('w', 'f4', ('time', 'z', 'r'))

# 写入属性
var_r.units = 'km'
var_r.long_name = 'radial distance'

var_z.units = 'km'
var_z.long_name = 'height (ASL)'

var_time.units = 'seconds since simulation start'
var_time.long_name = 'time'

var_u.units = 'm s-1'
var_u.long_name = 'radial velocity (outward positive)'

var_v.units = 'm s-1'
var_v.long_name = 'tangential velocity (cyclonic positive)'

var_w.units = 'm s-1'
var_w.long_name = 'vertical velocity'

# 写入固定坐标数据
var_z[:] = zh
var_r[:] = r_centers

print("输出文件结构初始化完成。")

检测到总时间步数: 961, 高度层数: 59
输出文件结构初始化完成。


In [3]:
# 3. 主循环：时间步遍历与中心定位
# 重新打开输入文件用于读取数据
nc_in = Dataset(nc_file, 'r')
xh = nc_in.variables['xh'][:]
yh = nc_in.variables['yh'][:]
time_var = nc_in.variables['time']

# 创建 2D 网格 (km) - 网格坐标是固定的，但相对于中心的距离 R 会随中心变化
X, Y = np.meshgrid(xh, yh)

print("开始批量处理...")

try:
    for t_idx in range(total_times):
        current_time = time_var[t_idx]
        
        # -------------------------------------------------------
        # 3.1 动态确定台风中心
        # -------------------------------------------------------
        try:
            # 调用 find_smoothed_min_point 寻找当前时刻的中心
            # 注意：这里传入 t_idx 作为 time_key
            center_res = find_smoothed_min_point(
                nc_file,
                time_key=t_idx,
                var_name="psfc",
                x_dim="xh",
                y_dim="yh",
                window=21,
                verbose=False # 关闭详细输出以免刷屏
            )
            center_x = center_res['x']
            center_y = center_res['y']
        except Exception as e:
            print(f"[Time Index {t_idx}] 中心定位失败: {e}。使用默认 (0,0)")
            center_x, center_y = 0.0, 0.0
            
        if t_idx % 10 == 0: # 每10步打印一次进度
            print(f"Processing Step {t_idx}/{total_times} (Time: {current_time:.1f}s) - Center: ({center_x:.2f}, {center_y:.2f})")

        # -------------------------------------------------------
        # 4. 数据读取与网格去交错处理
        # -------------------------------------------------------
        # 只读取当前时间步的数据
        u_stag = nc_in.variables['u'][t_idx, :, :, :] 
        v_stag = nc_in.variables['v'][t_idx, :, :, :]
        w_stag = nc_in.variables['w'][t_idx, :, :, :]

        # CM1 C-grid 去交错
        # u(zh, yh, xf) -> u(zh, yh, xh)
        if u_stag.shape[2] == len(xh) + 1:
            u_destag = 0.5 * (u_stag[:, :, :-1] + u_stag[:, :, 1:])
        else:
            u_destag = 0.5 * (u_stag[:, :, :-1] + u_stag[:, :, 1:])

        # v(zh, yf, xh) -> v(zh, yh, xh)
        if v_stag.shape[1] == len(yh) + 1:
            v_destag = 0.5 * (v_stag[:, :-1, :] + v_stag[:, 1:, :])
        else:
            v_destag = 0.5 * (v_stag[:, :-1, :] + v_stag[:, 1:, :])

        # w(zf, yh, xh) -> w(zh, yh, xh)
        if w_stag.shape[0] == len(zh) + 1:
            w_destag = 0.5 * (w_stag[:-1, :, :] + w_stag[1:, :, :])
        else:
            w_destag = 0.5 * (w_stag[:-1, :, :] + w_stag[1:, :, :])

        # -------------------------------------------------------
        # 5. 坐标转换与风场投影
        # -------------------------------------------------------
        # 计算相对于当前台风中心的距离 R 和角度 Theta
        R_2d = np.sqrt((X - center_x)**2 + (Y - center_y)**2)
        Theta_2d = np.arctan2(Y - center_y, X - center_x)

        # 扩展维度以匹配高度层 (nz, ny, nx)
        Theta_3d = np.tile(Theta_2d[np.newaxis, :, :], (nz, 1, 1))

        # 投影风速
        # Vr = u * cos(theta) + v * sin(theta)
        # Vt = -u * sin(theta) + v * cos(theta) (逆时针/气旋式为正)
        vr_3d = u_destag * np.cos(Theta_3d) + v_destag * np.sin(Theta_3d)
        vt_3d = -u_destag * np.sin(Theta_3d) + v_destag * np.cos(Theta_3d)
        
        # -------------------------------------------------------
        # 6. 执行方位角平均并写入文件
        # -------------------------------------------------------
        # 计算 bin 索引
        bin_indices = np.digitize(R_2d.ravel(), r_bins)
        
        # 临时数组存放当前时刻结果
        u_avg_t = np.zeros((nz, nr), dtype=np.float32)
        v_avg_t = np.zeros((nz, nr), dtype=np.float32)
        w_avg_t = np.zeros((nz, nr), dtype=np.float32)

        for k in range(nz):
            vr_flat = vr_3d[k, :, :].ravel()
            vt_flat = vt_3d[k, :, :].ravel()
            w_flat = w_destag[k, :, :].ravel()
            
            # 快速求和与计数
            count = np.bincount(bin_indices, minlength=len(r_bins)+1)
            sum_vr = np.bincount(bin_indices, weights=vr_flat, minlength=len(r_bins)+1)
            sum_vt = np.bincount(bin_indices, weights=vt_flat, minlength=len(r_bins)+1)
            sum_w  = np.bincount(bin_indices, weights=w_flat, minlength=len(r_bins)+1)
            
            with np.errstate(divide='ignore', invalid='ignore'):
                avg_vr_layer = sum_vr / count
                avg_vt_layer = sum_vt / count
                avg_w_layer  = sum_w / count
            
            # 取出有效范围 (索引 1 到 nr)
            u_avg_t[k, :] = avg_vr_layer[1:nr+1]
            v_avg_t[k, :] = avg_vt_layer[1:nr+1]
            w_avg_t[k, :] = avg_w_layer[1:nr+1]

        # 处理 NaN
        u_avg_t = np.nan_to_num(u_avg_t)
        v_avg_t = np.nan_to_num(v_avg_t)
        w_avg_t = np.nan_to_num(w_avg_t)

        # 写入 NetCDF (追加模式)
        var_time[t_idx] = current_time
        var_u[t_idx, :, :] = u_avg_t
        var_v[t_idx, :, :] = v_avg_t
        var_w[t_idx, :, :] = w_avg_t
        
        # 定期刷新缓冲区
        if t_idx % 5 == 0:
            nc_out.sync()

finally:
    # 确保文件关闭
    nc_in.close()
    nc_out.close()
    print("处理完成，文件已关闭。")

开始批量处理...
Processing Step 0/961 (Time: 0.0s) - Center: (-1.50, -1.50)
Processing Step 10/961 (Time: 9000.0s) - Center: (1.50, -1.50)
Processing Step 20/961 (Time: 18000.0s) - Center: (1.50, 1.50)
Processing Step 30/961 (Time: 27000.0s) - Center: (1.50, 1.50)
Processing Step 40/961 (Time: 36000.0s) - Center: (-1.50, -1.50)
Processing Step 50/961 (Time: 45000.0s) - Center: (4.50, -1.50)
Processing Step 60/961 (Time: 54000.0s) - Center: (4.50, 4.50)
Processing Step 70/961 (Time: 63000.0s) - Center: (-4.50, 4.50)
Processing Step 80/961 (Time: 72000.0s) - Center: (-7.50, -1.50)
Processing Step 90/961 (Time: 81000.0s) - Center: (-1.50, -10.50)
Processing Step 100/961 (Time: 90000.0s) - Center: (4.50, -7.50)
Processing Step 110/961 (Time: 99000.0s) - Center: (-16.50, -16.50)
Processing Step 120/961 (Time: 108000.0s) - Center: (1.50, -7.50)
Processing Step 130/961 (Time: 117000.0s) - Center: (-4.50, -1.50)
Processing Step 140/961 (Time: 126000.0s) - Center: (13.50, 7.50)
Processing Step 150/96